# 🎬 Reco-Spark ALS Model Training Pipeline

**MovieLens 25M** dataset üzerinde **PySpark ALS** modeli eğitimi.

| Adım | Açıklama |
|------|----------|
| 1 | Ortam Kurulumu (PySpark + Java) |
| 2 | Dataset İndirme & Yükleme |
| 3 | EDA (Keşifsel Veri Analizi) |
| 4 | Veri Temizleme |
| 5 | Train/Test Split (%80/%20) |
| 6 | ALS + CrossValidator Hyperparameter Tuning |
| 7 | Model Değerlendirme (RMSE < 1.0) |
| 8 | Model Export & İndirme |

---
## 📦 Adım 1: Ortam Kurulumu

In [ ]:
# PySpark kurulumu
!pip install pyspark==3.5.4 -q

In [ ]:
# Java kontrolü (Colab'da genelde yüklü gelir)
!java -version

In [ ]:
import os
import time
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType, StringType
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

print("Tüm importlar başarılı! ✅")

---
## 📥 Adım 2: Dataset İndirme & Yükleme

In [ ]:
# MovieLens 25M dataset indir
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
DATA_DIR = "/content/data"
EXTRACT_DIR = f"{DATA_DIR}/ml-25m"

if not os.path.exists(f"{EXTRACT_DIR}/ratings.csv"):
    print("📥 MovieLens 25M indiriliyor... (~250MB)")
    !mkdir -p {DATA_DIR}
    !wget -q --show-progress -O {DATA_DIR}/ml-25m.zip {DATASET_URL}
    print("📦 Arşiv açılıyor...")
    !unzip -q -o {DATA_DIR}/ml-25m.zip -d {DATA_DIR}
    !rm {DATA_DIR}/ml-25m.zip
    print("✅ Dataset hazır!")
else:
    print("✅ Dataset zaten mevcut!")

# Dosya boyutları
for f in os.listdir(EXTRACT_DIR):
    path = os.path.join(EXTRACT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  📄 {f}: {size_mb:.1f} MB")

In [ ]:
RATINGS_PATH = f"{EXTRACT_DIR}/ratings.csv"
MOVIES_PATH = f"{EXTRACT_DIR}/movies.csv"
MODEL_SAVE_DIR = "/content/als_model"

# SparkSession oluştur
spark = (
    SparkSession.builder
    .appName("Reco-Spark-ALS-Training")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"✅ SparkSession hazır! (version: {spark.version})")

---
## 🔍 Adım 3: EDA (Keşifsel Veri Analizi)

In [ ]:
# Schema tanımla
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", FloatType(), False),
    StructField("timestamp", LongType(), True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField("genres", StringType(), True),
])

# Veri yükle
print("📂 Ratings yükleniyor...")
ratings = spark.read.option("header", "true").schema(ratings_schema).csv(RATINGS_PATH)

print("📂 Movies yükleniyor...")
movies = spark.read.option("header", "true").schema(movies_schema).csv(MOVIES_PATH)

ratings.cache()
movies.cache()

print("✅ Veriler yüklendi!")

In [ ]:
# Schema bilgisi
print("📋 Ratings Schema:")
ratings.printSchema()
print("📋 Movies Schema:")
movies.printSchema()

In [ ]:
# Boyut ve istatistikler
rating_count = ratings.count()
user_count = ratings.select("userId").distinct().count()
unique_movies = ratings.select("movieId").distinct().count()
movie_count = movies.count()

print(f"📊 Toplam rating sayısı      : {rating_count:,}")
print(f"📊 Benzersiz kullanıcı sayısı : {user_count:,}")
print(f"📊 Benzersiz film sayısı      : {unique_movies:,}")
print(f"📊 Movies tablosu satır sayısı: {movie_count:,}")

In [ ]:
# Rating dağılımı
print("📊 Rating Dağılımı:")
ratings.groupBy("rating").count().orderBy("rating").show()

print("📊 Rating İstatistikleri:")
ratings.select("rating").describe().show()

In [ ]:
# Örnek veriler
print("📊 Örnek Ratings:")
ratings.show(5, truncate=False)

print("📊 Örnek Movies:")
movies.show(5, truncate=False)

---
## 🧹 Adım 4: Veri Temizleme

In [ ]:
before = ratings.count()
ratings_clean = ratings.dropna(subset=["userId", "movieId", "rating"])
after = ratings_clean.count()

print(f"  Temizleme öncesi : {before:,} satır")
print(f"  Temizleme sonrası: {after:,} satır")
print(f"  Silinen satır    : {before - after:,}")

# ALS için sadece gerekli sütunlar
ratings_clean = ratings_clean.select("userId", "movieId", "rating")
print("\n✅ Veri temizlendi!")

---
## ✂️ Adım 5: Train/Test Split (%80 / %20)

In [ ]:
train, test = ratings_clean.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

train_count = train.count()
test_count = test.count()
total = train_count + test_count

print(f"  Train set: {train_count:,} satır ({train_count/total*100:.1f}%)")
print(f"  Test set : {test_count:,} satır ({test_count/total*100:.1f}%)")
print("\n✅ Train/Test split tamamlandı!")

---
## 🤖 Adım 6: ALS + CrossValidator Hyperparameter Tuning

**Grid:**
- `rank`: [10, 50, 100]
- `regParam`: [0.01, 0.1, 1.0]
- `maxIter`: [5, 10, 20]

Toplam: **27 kombinasyon × 3 fold = 81 model**

In [ ]:
# ALS tanımı
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="drop",
)

# Hyperparameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank, [10, 50, 100])
    .addGrid(als.regParam, [0.01, 0.1, 1.0])
    .addGrid(als.maxIter, [5, 10, 20])
    .build()
)

# Evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction",
)

# CrossValidator
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"3-fold CV ile toplam: {len(param_grid) * 3} model eğitilecek")
print("\n⏳ CrossValidator başlatılıyor... (Bu 30-90 dakika sürebilir)")

start_time = time.time()
cv_model = cv.fit(train)
elapsed = time.time() - start_time

print(f"\n✅ CrossValidator tamamlandı! Süre: {elapsed/60:.1f} dakika")

---
## 📈 Adım 7: Model Değerlendirme

In [ ]:
best_model = cv_model.bestModel
predictions = best_model.transform(test)
rmse = evaluator.evaluate(predictions)

print(f"📈 Test RMSE: {rmse:.4f}")
print(f"🎯 Hedef:     RMSE < 1.0")
print(f"{'✅ HEDEF BAŞARILDI!' if rmse < 1.0 else '❌ HEDEF BAŞARILAMADI'}")

# En iyi parametreler
print(f"\n🏆 En İyi Parametreler:")
print(f"  rank     = {best_model.rank}")
print(f"  maxIter  = {best_model._java_obj.parent().getMaxIter()}")
print(f"  regParam = {best_model._java_obj.parent().getRegParam()}")

In [ ]:
# Cross Validation sonuçları - tüm kombinasyonlar
avg_metrics = cv_model.avgMetrics

print("📊 Cross Validation Sonuçları (RMSE):")
print("-" * 50)

results = []
for i, (params, metric) in enumerate(zip(param_grid, avg_metrics)):
    rank_val = params[als.rank]
    reg_val = params[als.regParam]
    iter_val = params[als.maxIter]
    results.append((i+1, rank_val, reg_val, iter_val, metric))
    print(f"  #{i+1:2d}: rank={rank_val:3d}, regParam={reg_val:.2f}, maxIter={iter_val:2d} → RMSE={metric:.4f}")

# En iyi kombinasyon
best_idx = avg_metrics.index(min(avg_metrics))
print(f"\n🏆 En iyi: #{best_idx+1} (RMSE = {min(avg_metrics):.4f})")

In [ ]:
# Kullanıcı 1 için örnek öneriler
print("🎬 Kullanıcı 1 için Top-10 Film Önerisi:")
print("-" * 60)

user_recs = best_model.recommendForAllUsers(10)
user1_recs = user_recs.filter(user_recs.userId == 1).collect()

if user1_recs:
    for j, rec in enumerate(user1_recs[0].recommendations, 1):
        movie_row = movies.filter(movies.movieId == rec.movieId).collect()
        title = movie_row[0].title if movie_row else "Bilinmeyen"
        genres = movie_row[0].genres if movie_row else "-"
        print(f"  {j:2d}. {title}")
        print(f"      Tür: {genres} | Tahmini Puan: {rec.rating:.2f}")

---
## 💾 Adım 8: Model Export & İndirme

In [ ]:
# Modeli kaydet
if os.path.exists(MODEL_SAVE_DIR):
    shutil.rmtree(MODEL_SAVE_DIR)

best_model.write().overwrite().save(MODEL_SAVE_DIR)
print(f"💾 Model kaydedildi: {MODEL_SAVE_DIR}")

# Metadata kaydet
metadata_path = "/content/model_metadata.txt"
with open(metadata_path, "w") as f:
    f.write(f"Model: ALS (PySpark 3.5.4)\n")
    f.write(f"Dataset: MovieLens 25M\n")
    f.write(f"Test RMSE: {rmse:.4f}\n")
    f.write(f"Rank: {best_model.rank}\n")
    f.write(f"MaxIter: {best_model._java_obj.parent().getMaxIter()}\n")
    f.write(f"RegParam: {best_model._java_obj.parent().getRegParam()}\n")
    f.write(f"Training Time: {elapsed/60:.1f} minutes\n")
    f.write(f"Trained at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"\nAll CV Results (RMSE):\n")
    for i, (params, metric) in enumerate(zip(param_grid, avg_metrics)):
        f.write(f"  #{i+1}: rank={params[als.rank]}, regParam={params[als.regParam]}, maxIter={params[als.maxIter]} -> RMSE={metric:.4f}\n")

print(f"📝 Metadata kaydedildi: {metadata_path}")

In [ ]:
# Modeli zip olarak paketle (indirmek için)
ZIP_PATH = "/content/als_model_export"

!cp /content/model_metadata.txt {MODEL_SAVE_DIR}/
shutil.make_archive(ZIP_PATH, 'zip', MODEL_SAVE_DIR)

zip_size = os.path.getsize(f"{ZIP_PATH}.zip") / (1024*1024)
print(f"📦 Model arşivi: {ZIP_PATH}.zip ({zip_size:.1f} MB)")
print("\n⬇️ Aşağıdaki hücreyi çalıştırarak modeli indirin!")

In [ ]:
# Modeli indir
from google.colab import files

print("⬇️ Model indirme başlatılıyor...")
files.download(f"{ZIP_PATH}.zip")
files.download(metadata_path)
print("✅ İndirme tamamlandı!")

In [ ]:
# Doğrulama: Modeli tekrar yükleyip test et
print("🔄 Model doğrulama - yeniden yükleniyor...")
loaded_model = ALSModel.load(MODEL_SAVE_DIR)
loaded_predictions = loaded_model.transform(test)
loaded_rmse = evaluator.evaluate(loaded_predictions)
print(f"📈 Yüklenen model RMSE: {loaded_rmse:.4f}")
print(f"📈 Orijinal RMSE:       {rmse:.4f}")
print(f"{'✅ Model doğrulandı!' if abs(loaded_rmse - rmse) < 0.0001 else '⚠️ RMSE farkı var!'}")

In [ ]:
# Spark oturumunu kapat
spark.stop()
print("\n🔌 Spark oturumu kapatıldı.")
print("\n" + "=" * 60)
print("  🎉 Pipeline tamamlandı!")
print("=" * 60)
print(f"\n  📈 Final RMSE: {rmse:.4f}")
print(f"  🏆 Best Rank: {best_model.rank}")
print(f"  📦 Model dosyası: als_model_export.zip")
print(f"\n  Sonraki adım: ZIP dosyasını indirip")
print(f"  ml_pipeline/models/als_model/ klasörüne çıkart.")